# Wan2GP on Google Colab

Sets up [Wan2GP](https://github.com/deepbeepmeep/Wan2GP) in a fresh GPU-backed Colab session. 

Run the cells in order to prepare the runtime, install dependencies, and launch the Gradio interface. Click on the link in the output from the last cell to launch the app in your browser.

> **Colab free-tier note:** free sessions commonly have a 15 GB T4 GPU and limited system RAM. This notebook defaults to a conservative-but-faster launch preset: fp16, SDPA attention, TeaCache, capped reserved RAM, Wan2GP profile 4, and the optional GGUF CUDA kernels that accelerate GGUF models. If you hit out-of-memory errors, switch `COLAB_FREE_TIER_PRESET` to `'minimum_memory'` in Step 2.


## 1. Confirm the accelerator

Choose `Runtime → Change runtime type` and select **GPU** before running anything else.

If this cell raises an error, go back to `Runtime → Change runtime type`, pick **GPU** and save.


In [ ]:
import subprocess

try:
    subprocess.run(['nvidia-smi'], check=True)
except Exception as exc:
    raise RuntimeError(
        'GPU not detected. In Colab, open Runtime → Change runtime type, select GPU (or TPU if GPUs are unavailable), save, then rerun this cell.'
    ) from exc


## 2. Configure storage and the Colab performance preset

Set `USE_GOOGLE_DRIVE_DATA = True` if you want Google Drive to keep checkpoints, LoRAs, outputs and model caches across Colab restarts. Runtime disk is usually faster, so leave it off when you only care about speed for the current session.

`COLAB_FREE_TIER_PRESET = 'balanced_fast'` keeps RAM/VRAM use low enough for a typical free T4 while avoiding the extra offload overhead of Wan2GP profile 5. Use `'minimum_memory'` only if the balanced preset still runs out of memory.


In [ ]:
from pathlib import Path


# CHANGE THIS TO TRUE IF YOU WANT TO USE GOOGLE DRIVE FOR DATA STORAGE (PERSISTENT ACROSS SESSIONS).
# Runtime disk is faster than Drive for active caches, so False is best for speed.
USE_GOOGLE_DRIVE_DATA = False

# Free-tier speed/VRAM preset. Options: 'balanced_fast' or 'minimum_memory'.
COLAB_FREE_TIER_PRESET = 'balanced_fast'
COLAB_PRESETS = {
    'balanced_fast': {
        'profile': '4',
        'attention': 'sdpa',
        'teacache': '2.0',
        'reserved_ram': '0.20',
        'fp16': True,
        'preload_mb': '0',
        'description': 'Faster free-tier default: profile 4, fp16, TeaCache 2.0, and a small RAM reserve.',
    },
    'minimum_memory': {
        'profile': '5',
        'attention': 'sdpa',
        'teacache': '1.5',
        'reserved_ram': '0.10',
        'fp16': True,
        'preload_mb': '0',
        'description': 'Lowest RAM/VRAM fallback: more offloading and slightly slower generation.',
    },
}

if COLAB_FREE_TIER_PRESET not in COLAB_PRESETS:
    raise ValueError(f'Unknown COLAB_FREE_TIER_PRESET: {COLAB_FREE_TIER_PRESET}. Choose one of {tuple(COLAB_PRESETS)}.')

WAN2GP_LAUNCH_PRESET = COLAB_PRESETS[COLAB_FREE_TIER_PRESET]

DRIVE_MOUNT_POINT = Path('/content/drive')
WAN2GP_ROOT = Path('/content/Wan2GP').resolve()
EPHEMERAL_DATA_ROOT = Path('/content/Wan2GP-data').resolve()
PERSISTENT_DATA_ROOT = (DRIVE_MOUNT_POINT / 'MyDrive' / 'Wan2GP-data').resolve()

if USE_GOOGLE_DRIVE_DATA:
    from google.colab import drive

    drive.mount(str(DRIVE_MOUNT_POINT), force_remount=False)
    WAN_DATA_ROOT = PERSISTENT_DATA_ROOT
    data_mode = 'Google Drive (persistent data)'
else:
    WAN_DATA_ROOT = EPHEMERAL_DATA_ROOT
    data_mode = 'Colab runtime disk (ephemeral data, fastest)'

WAN_CKPTS_DIR = (WAN_DATA_ROOT / 'ckpts').resolve()
WAN_LORAS_DIR = (WAN_DATA_ROOT / 'loras').resolve()
WAN_OUTPUTS_DIR = (WAN_DATA_ROOT / 'outputs').resolve()
WAN_CACHE_DIR = (WAN_DATA_ROOT / 'cache').resolve()
WAN_CONFIG_DIR = (WAN_DATA_ROOT / 'config').resolve()
WAN_LTX2_LORAS_DIR = (WAN_LORAS_DIR / 'ltx2').resolve()
WAN_LTX2_22B_LORAS_DIR = (WAN_LORAS_DIR / 'ltx2_22B').resolve()

for directory in (WAN2GP_ROOT.parent, WAN_DATA_ROOT, WAN_CKPTS_DIR, WAN_LORAS_DIR, WAN_OUTPUTS_DIR, WAN_CACHE_DIR, WAN_CONFIG_DIR, WAN_LTX2_LORAS_DIR, WAN_LTX2_22B_LORAS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f'Wan2GP repository path: {WAN2GP_ROOT}')
print(f'Data storage mode: {data_mode}')
print(f'Data root: {WAN_DATA_ROOT}')
print(f'Checkpoints: {WAN_CKPTS_DIR}')
print(f'LoRAs: {WAN_LORAS_DIR}')
print(f'LTX-2 LoRAs: {WAN_LTX2_LORAS_DIR}')
print(f'LTX-2 22B LoRAs: {WAN_LTX2_22B_LORAS_DIR}')
print(f'Outputs: {WAN_OUTPUTS_DIR}')
print(f'Cache: {WAN_CACHE_DIR}')
print(f'Config: {WAN_CONFIG_DIR}')
print(f"Launch preset: {COLAB_FREE_TIER_PRESET} - {WAN2GP_LAUNCH_PRESET['description']}")


## 3. Download or update Wan2GP

Clone the repository if it is not present yet; otherwise pull the latest changes.


In [ ]:
import shutil, subprocess
from pathlib import Path

def merge_directory_contents(source_dir: Path, destination_dir: Path) -> None:
    for child in list(source_dir.iterdir()):
        destination = destination_dir / child.name
        if destination.exists():
            if child.is_dir() and destination.is_dir():
                merge_directory_contents(child, destination)
                child.rmdir()
                continue
            raise RuntimeError(f'Cannot move {child} into {destination_dir}: {destination} already exists.')
        shutil.move(str(child), str(destination))

def attach_data_directory(repo_path: Path, data_path: Path) -> None:
    data_path.mkdir(parents=True, exist_ok=True)

    if repo_path.is_symlink():
        if repo_path.resolve() != data_path.resolve():
            raise RuntimeError(f'{repo_path} already points to {repo_path.resolve()}, expected {data_path}.')
        print(f'Using existing link: {repo_path} -> {data_path}')
        return

    if repo_path.exists():
        if not repo_path.is_dir():
            raise RuntimeError(f'Expected a directory at {repo_path}.')
        merge_directory_contents(repo_path, data_path)
        repo_path.rmdir()
    else:
        repo_path.parent.mkdir(parents=True, exist_ok=True)

    repo_path.symlink_to(data_path, target_is_directory=True)
    print(f'Linked {repo_path.name} -> {data_path}')

repo_url = 'https://github.com/deepbeepmeep/Wan2GP.git'
if WAN2GP_ROOT.exists():
    status = subprocess.run(
        ['git', '-C', str(WAN2GP_ROOT), 'status', '--porcelain'],
        check=True,
        capture_output=True,
        text=True,
    )
    if status.stdout.strip():
        print('Repository already exists and has local changes. Skipping git pull to preserve your persistent files.')
    else:
        print('Repository already exists. Pulling latest changes...')
        subprocess.run(['git', '-C', str(WAN2GP_ROOT), 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', repo_url, str(WAN2GP_ROOT)], check=True)

attach_data_directory(WAN2GP_ROOT / 'ckpts', WAN_CKPTS_DIR)
attach_data_directory(WAN2GP_ROOT / 'loras', WAN_LORAS_DIR)
attach_data_directory(WAN2GP_ROOT / 'outputs', WAN_OUTPUTS_DIR)


## 4. Install system dependencies

Install shared libraries needed for video and audio processing. If you see a warning about skipping an extra repository, it is safe to ignore.

In [ ]:
import os, subprocess

env = os.environ.copy()
env['DEBIAN_FRONTEND'] = 'noninteractive'

subprocess.run(['sudo', 'apt-get', 'update', '-qq'], check=True, env=env)
subprocess.run([
    'sudo', 'apt-get', 'install', '-y', '--no-install-recommends',
    'ffmpeg', 'libglib2.0-0', 'libgl1', 'libportaudio2'
], check=True, env=env)


## 5. Install Python dependencies and optional GGUF CUDA kernels

Install PyTorch, xformers, Hugging Face's faster transfer helper, Wan2GP's Python packages, and the prebuilt `llamacpp_gguf_cuda` wheel used to accelerate GGUF models. The GGUF wheel is tied to specific Python/PyTorch/CUDA builds, so this notebook uses Wan2GP's CUDA 13 / Torch 2.10 stack on Python 3.11.


In [ ]:
import importlib, os, subprocess, sys

env = os.environ.copy()
env.setdefault('DEBIAN_FRONTEND', 'noninteractive')

PYTORCH_INDEX_URL = 'https://download.pytorch.org/whl/cu130'
TORCH_VERSION = '2.10.0'
TORCHVISION_VERSION = '0.25.0'
TORCHAUDIO_VERSION = '2.10.0'
GGUF_CUDA_KERNEL_WHEEL = 'https://github.com/deepbeepmeep/kernels/releases/download/GGUF_Kernels/llamacpp_gguf_cuda-1.0.2+torch210cu13py311-cp311-cp311-linux_x86_64.whl'
INSTALL_GGUF_CUDA_KERNELS = True

if sys.version_info[:2] != (3, 11):
    raise RuntimeError(
        f'This Colab notebook expects Python 3.11 for the prebuilt GGUF CUDA kernel wheel; found Python {sys.version_info.major}.{sys.version_info.minor}.'
    )

subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'], check=True, env=env)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torch', 'torchvision', 'torchaudio'], check=False, env=env)
subprocess.run([
    sys.executable,
    '-m',
    'pip',
    'install',
    '--no-cache-dir',
    '--force-reinstall',
    f'torch=={TORCH_VERSION}',
    f'torchvision=={TORCHVISION_VERSION}',
    f'torchaudio=={TORCHAUDIO_VERSION}',
    '--index-url',
    PYTORCH_INDEX_URL,
], check=True, env=env)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'xformers'], check=True, env=env)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', 'hf_transfer'], check=True, env=env)
if INSTALL_GGUF_CUDA_KERNELS:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', GGUF_CUDA_KERNEL_WHEEL], check=True, env=env)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(WAN2GP_ROOT / 'requirements.txt')], check=True, env=env)

installed = {name: importlib.import_module(name).__version__.split('+', 1)[0] for name in ('torch', 'torchvision', 'torchaudio')}
expected = {
    'torch': TORCH_VERSION,
    'torchvision': TORCHVISION_VERSION,
    'torchaudio': TORCHAUDIO_VERSION,
}
mismatches = {name: version for name, version in installed.items() if version != expected[name]}
if mismatches:
    raise RuntimeError(f'PyTorch package mismatch after install: {mismatches}. Restart the Colab runtime and rerun Step 5.')

if INSTALL_GGUF_CUDA_KERNELS:
    try:
        import llamacpp_gguf_cuda  # noqa: F401
    except ImportError as exc:
        raise RuntimeError('GGUF CUDA kernels failed to import after installation. Restart the Colab runtime and rerun Step 5.') from exc
    print('Installed GGUF CUDA kernels: llamacpp_gguf_cuda')

print('Installed torch stack:', ', '.join(f"{name} {version}" for name, version in installed.items()))


## 5b. Force a headless matplotlib backend

Ensure Wan2GP's preprocessing tools use the headless Agg backend so Step 6 launches cleanly in Colab.


In [ ]:
from pathlib import Path

# Replace the TkAgg backend with the headless Agg backend if present.
target = WAN2GP_ROOT / 'preprocessing/matanyone/tools/interact_tools.py'
needle = "matplotlib.use('TkAgg')"
replacement = "matplotlib.use('Agg')"

if not target.exists():
    print(f'Skipping: {target} not found.')
else:
    text = target.read_text()
    if replacement in text:
        print('Agg backend already set; no change needed.')
    elif needle in text:
        target.write_text(text.replace(needle, replacement, 1))
        print('Replaced TkAgg with Agg in interact_tools.py.')
    else:
        print('Backend call not found; no change made.')


## 6. Launch Wan2GP

Run the Gradio interface. You will find the gradio link in the output. Click on the link to access the UI. Keep the cell running to stay connected; stop it with the square **Stop** button when you are finished.

The launch command is assembled from the Step 2 preset so you can trade speed for memory without editing the subprocess call.


In [ ]:
import os, subprocess, sys, threading, time

env = os.environ.copy()
env['WAN_CACHE_DIR'] = str(WAN_CACHE_DIR)
env['HF_HOME'] = str(WAN_CACHE_DIR / 'huggingface')
env['HUGGINGFACE_HUB_CACHE'] = str(WAN_CACHE_DIR / 'huggingface' / 'hub')
env['TRANSFORMERS_CACHE'] = str(WAN_CACHE_DIR / 'huggingface' / 'transformers')
env['TORCH_HOME'] = str(WAN_CACHE_DIR / 'torch')
env['XDG_CACHE_HOME'] = str(WAN_CACHE_DIR / '.cache')
env['TRITON_CACHE_DIR'] = str(WAN_CACHE_DIR / 'triton')
env['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,garbage_collection_threshold:0.8'

preset = WAN2GP_LAUNCH_PRESET
cmd = [
    sys.executable,
    '-u',
    'wgp.py',
    '--listen',
    '--server-port', '7860',
    '--share',
    '--profile', preset['profile'],
    '--attention', preset['attention'],
    '--teacache', preset['teacache'],
    '--perc-reserved-mem-max', preset['reserved_ram'],
    '--preload', preset['preload_mb'],
    '--config', str(WAN_CONFIG_DIR),
]
if preset.get('fp16'):
    cmd.append('--fp16')

if USE_GOOGLE_DRIVE_DATA:
    print('Using Google Drive for checkpoints, LoRAs, outputs and caches.')
else:
    print('Using Colab runtime storage for checkpoints, LoRAs, outputs and caches.')
print(f"Using launch preset: {COLAB_FREE_TIER_PRESET} - {preset['description']}")
print('Launch command:', ' '.join(cmd))
print('Launching Wan2GP…')
process = subprocess.Popen(
    cmd,
    cwd=str(WAN2GP_ROOT),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
stop_event = threading.Event()

def keepalive():
    while not stop_event.is_set():
        time.sleep(45)
        if stop_event.is_set():
            break
        print('[keepalive] Notebook cell still running…')

keepalive_thread = threading.Thread(target=keepalive, daemon=True)
keepalive_thread.start()

try:
    for line in iter(process.stdout.readline, ''):
        if not line:
            break
        print(line, end='')
except KeyboardInterrupt:
    print('Stopping Wan2GP…')
    process.terminate()
finally:
    stop_event.set()
    process.wait()
    keepalive_thread.join(timeout=1)
    print(f'Wan2GP stopped (return code: {process.returncode}).')
